# AIAMM V8 FRA Notebook

This notebook integrates lessons from V6FRA and V7FRA and adds V8 controls for reward stability, anti-thrashing, regime adaptation, and controlled benchmark fairness.

Execution order: run top to bottom.


## Section 0: Imports and Setup

Imports and global setup utilities used across all sections.


In [4]:
import numpy as np
import random
from collections import deque
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


In [5]:
import numpy as np
import random
from collections import deque
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Seeding
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def set_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# AMMEnvironment  (unchanged â€” microstructure is fine)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


## Section 1: AMM Environment

Defines the AMM microstructure (market feed, order flow, arbitrage, swap mechanics) with dedicated RNG support for controlled comparisons.


In [6]:
class AMMEnvironment:
    def __init__(
        self,
        initial_token_a=10_000,
        initial_token_b=10_000,
        fee=0.003,
        price_feed=None,
        arbitrage_band_bps=12,
        max_trade_fraction=0.12,
        arb_gas_cost_b=0.25,
        rebalance_gas_cost_b=0.15,
        rng=None,
    ):
        self.initial_token_a = initial_token_a
        self.initial_token_b = initial_token_b
        self.token_a = initial_token_a
        self.token_b = initial_token_b
        self.fee = fee
        self.max_trade_fraction = max_trade_fraction
        self.arbitrage_band_bps = arbitrage_band_bps
        self.arb_gas_cost_b = arb_gas_cost_b
        self.rebalance_gas_cost_b = rebalance_gas_cost_b
        self.market_price = self.get_price()
        self.price_history = [self.get_price()]
        self.market_price_history = [self.market_price]
        self.volume_history = []
        self.fee_history = []
        self.arb_volume_history = []
        self.dislocation_history = [0.0]
        self.step_count = 0
        self.price_feed = price_feed
        self.price_feed_index = 0
        self.market_volatility = 0.015
        self.market_drift = 0.0
        self.rng = rng if rng is not None else np.random.default_rng()

    def get_price(self):
        return self.token_b / self.token_a if self.token_a > 0 else float("inf")

    def _next_market_price(self, sentiment=0.0):
        if self.price_feed:
            if self.price_feed_index < len(self.price_feed):
                p = self.price_feed[self.price_feed_index]
                self.price_feed_index += 1
                if p > 0:
                    self.market_price = float(p)
                    return self.market_price

        if self.step_count > 0 and self.step_count % 120 == 0:
            self.market_volatility = float(self.rng.choice([0.008, 0.015, 0.03]))
            self.market_drift = float(self.rng.normal(0, 0.0015))

        shock = self.rng.normal(
            self.market_drift + sentiment * 0.003,
            self.market_volatility,
        )
        self.market_price = max(1e-6, self.market_price * np.exp(shock))
        return self.market_price

    def _simulate_order_flow(self, sentiment=0.0):
        base = self.token_b * 0.0025
        notional = base * self.rng.lognormal(0.0, 0.65)
        buy_prob = np.clip(0.5 + 0.35 * np.tanh(sentiment), 0.05, 0.95)
        if self.rng.random() < buy_prob:
            _, vol_b, fee_b = self.swap_b_for_a(notional)
        else:
            amount_a = notional / max(self.market_price, 1e-9)
            _, vol_b, fee_b = self.swap_a_for_b(amount_a)
        return vol_b, fee_b

    def _run_arbitrage(self):
        amm = self.get_price()
        mkt = self.market_price
        if not np.isfinite(amm) or mkt <= 0:
            return 0.0, 0.0
        disloc = (amm - mkt) / mkt
        if abs(disloc) * 10_000 < self.arbitrage_band_bps:
            return 0.0, 0.0
        target = amm + 0.8 * (mkt - amm)
        k = self.token_a * self.token_b
        if target > amm:
            db = max(0.0, np.sqrt(k * target) - self.token_b)
            amt = db / max(1 - self.fee, 1e-9)
            if amt < self.arb_gas_cost_b * 8:
                return 0.0, 0.0
            _, vol_b, fee_b = self.swap_b_for_a(amt)
        else:
            da = max(0.0, np.sqrt(k / max(target, 1e-9)) - self.token_a)
            amt = da / max(1 - self.fee, 1e-9)
            if amt * max(self.market_price, 1e-9) < self.arb_gas_cost_b * 8:
                return 0.0, 0.0
            _, vol_b, fee_b = self.swap_a_for_b(amt)
        return vol_b, fee_b

    def simulate_market_activity(self, sentiment=0.0):
        self.step_count += 1
        self._next_market_price(sentiment)
        flow_vol, flow_fee = self._simulate_order_flow(sentiment)
        arb_vol, arb_fee = self._run_arbitrage()
        total_vol = flow_vol + arb_vol
        total_fee = flow_fee + arb_fee
        amm = self.get_price()
        disloc = (amm - self.market_price) / max(self.market_price, 1e-9)
        self.price_history.append(amm)
        self.market_price_history.append(self.market_price)
        self.volume_history.append(total_vol)
        self.fee_history.append(total_fee)
        self.arb_volume_history.append(arb_vol)
        self.dislocation_history.append(disloc)
        return {
            "market_price": self.market_price,
            "amm_price": amm,
            "price_dislocation": disloc,
            "volume_b": total_vol,
            "fees_collected_b": total_fee,
            "arb_volume_b": arb_vol,
        }

    def swap_a_for_b(self, amount_a):
        if amount_a <= 0:
            return 0.0, 0.0, 0.0
        amount_a = min(amount_a, self.token_a * self.max_trade_fraction)
        net = amount_a * (1 - self.fee)
        out_b = self.token_b * net / (self.token_a + net)
        fee_a = amount_a - net
        self.token_a += amount_a
        self.token_b -= out_b
        return out_b, amount_a * max(self.market_price, 1e-9), fee_a * max(self.market_price, 1e-9)

    def swap_b_for_a(self, amount_b):
        if amount_b <= 0:
            return 0.0, 0.0, 0.0
        amount_b = min(amount_b, self.token_b * self.max_trade_fraction)
        net = amount_b * (1 - self.fee)
        out_a = self.token_a * net / (self.token_b + net)
        fee_b = amount_b - net
        self.token_b += amount_b
        self.token_a -= out_a
        return out_a, amount_b, fee_b

    def reset(self):
        self.token_a = self.initial_token_a
        self.token_b = self.initial_token_b
        self.market_price = self.get_price()
        self.price_history = [self.get_price()]
        self.market_price_history = [self.market_price]
        self.volume_history = []
        self.fee_history = []
        self.arb_volume_history = []
        self.dislocation_history = [0.0]
        self.step_count = 0
        self.price_feed_index = 0


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# LiquidityProvider
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


## Section 2: Liquidity Provider Accounting

Defines LP value accounting, impermanent loss, fee accrual, and active exposure valuation.


In [7]:
class LiquidityProvider:
    def __init__(self, deposit_a, deposit_b, entry_price):
        self.deposit_a = deposit_a
        self.deposit_b = deposit_b
        self.entry_price = entry_price
        self.initial_value = deposit_a * entry_price + deposit_b
        self.fees_earned = 0.0
        self.costs_paid = 0.0

    def calculate_impermanent_loss(self, current_price):
        if self.entry_price <= 0:
            return 0.0
        r = current_price / self.entry_price
        return 2 * np.sqrt(r) / (1 + r) - 1 if r > 0 else 0.0

    def hodl_value(self, current_price):
        return self.deposit_a * current_price + self.deposit_b

    def calculate_position_value(self, current_price, active_exposure=1.0, trading_costs=0.0):
        active_exposure = float(np.clip(active_exposure, 0.0, 1.0))
        il = self.calculate_impermanent_loss(current_price) * active_exposure
        hodl = self.hodl_value(current_price)
        return hodl * (1 + il) + self.fees_earned - trading_costs

    def add_fees(self, amount):
        self.fees_earned += max(0.0, amount)

    def add_cost(self, amount):
        self.costs_paid += max(0.0, amount)


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# SentimentAnalyzer
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


## Section 3: Sentiment Model

Provides sentiment generation from feed or stochastic logic, used as an exogenous market-flow driver.


In [8]:
class SentimentAnalyzer:
    def __init__(self, sentiment_feed=None, rng=None):
        self.sentiment_history = []
        self.base_sentiment = 0.0
        self.sentiment_feed = list(sentiment_feed) if sentiment_feed is not None else None
        self.sentiment_feed_index = 0
        self.rng = rng if rng is not None else np.random.default_rng()

    def get_sentiment(self, price_history, window=10):
        if self.sentiment_feed is not None:
            if self.sentiment_feed_index < len(self.sentiment_feed):
                s = float(self.sentiment_feed[self.sentiment_feed_index])
                self.sentiment_feed_index += 1
            else:
                s = 0.0
        elif len(price_history) < 2:
            s = self.rng.normal(0, 0.1)
        else:
            recent = price_history[-min(window, len(price_history)):]
            rets = [(recent[i]-recent[i-1])/recent[i-1]
                    for i in range(1, len(recent)) if recent[i-1] != 0]
            if rets:
                mom = np.mean(rets)
                vol = np.std(rets) if len(rets) > 1 else 0.01
                s = np.tanh(mom / max(vol, 0.001) * 5) + self.rng.normal(0, 0.2)
            else:
                s = self.rng.normal(0, 0.1)
        self.base_sentiment = 0.7 * self.base_sentiment + 0.3 * s
        final = float(np.clip(self.base_sentiment, -1, 1))
        self.sentiment_history.append(final)
        return final


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# RunningNormalizer
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


## Section 4: RL Core (Normalizer, Network, Agent)

Implements the V8 RL agent: normalized state, shallow network Q approximator, bounded risk-adjusted reward, and anti-thrashing action logic.


In [9]:
class RunningNormalizer:
    """Welford online z-score normaliser."""
    def __init__(self, size, clip=5.0):
        self.n = 0
        self.mean = np.zeros(size)
        self.M2 = np.ones(size)
        self.clip = clip

    def update_and_normalize(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        self.M2 += delta * (x - self.mean)
        if self.n < 2:
            return np.zeros_like(x)
        std = np.sqrt(self.M2 / (self.n - 1))
        std = np.where(std < 1e-8, 1.0, std)
        return np.clip((x - self.mean) / std, -self.clip, self.clip)


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# ShallowNet â€” one hidden layer Q-network  (FIX: nonlinear approximator)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class ShallowNet:
    """
    One hidden layer: input â†’ tanh â†’ output
    state_size â†’ hidden_size â†’ action_size
    Gradient descent on MSE with minibatch.
    """
    def __init__(self, state_size, action_size, hidden_size=32, lr=0.001, rng=None):
        self.rng = rng if rng is not None else np.random.default_rng()
        scale = np.sqrt(2.0 / state_size)
        self.W1 = self.rng.normal(0.0, scale, size=(state_size, hidden_size))
        self.b1 = np.zeros(hidden_size)
        self.W2 = self.rng.normal(0.0, 0.01, size=(hidden_size, action_size))
        self.b2 = np.zeros(action_size)
        self.lr = lr

    def _forward(self, x):
        h = np.tanh(x @ self.W1 + self.b1)
        return h, h @ self.W2 + self.b2

    def predict(self, x):
        _, q = self._forward(x)
        return q

    def update(self, x, action, target):
        h, q = self._forward(x)
        error = target - q[action]
        # Output layer gradient
        dW2 = np.outer(h, np.zeros(self.W2.shape[1]))
        dW2[:, action] = h * error
        db2 = np.zeros(self.b2.shape)
        db2[action] = error
        # Hidden layer gradient (chain rule through tanh)
        dh = error * self.W2[:, action]
        dtanh = (1 - h**2) * dh
        dW1 = np.outer(x, dtanh)
        db1 = dtanh
        # Update
        self.W1 += self.lr * dW1
        self.b1 += self.lr * db1
        self.W2 += self.lr * dW2
        self.b2 += self.lr * db2

    def copy_weights_from(self, other):
        self.W1 = other.W1.copy()
        self.b1 = other.b1.copy()
        self.W2 = other.W2.copy()
        self.b2 = other.b2.copy()


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# DQLAgent V7
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class DQLAgent:
    """V8 RL agent with bounded reward and anti-thrash controls."""

    ACTION_NAMES = [
        "Hold Position",
        "Increase Liquidity",
        "Decrease Liquidity",
        "Recenter Range Up",
        "Recenter Range Down",
    ]
    N_ACTIONS = 5

    # Tighter liquidity floor to avoid reward hacking through near-withdrawal.
    MIN_LIQ = 0.7
    MAX_LIQ = 1.0

    def __init__(
        self,
        state_size=14,
        hidden_size=48,
        target_sync=160,
        memory_size=12000,
        seed=42,
    ):
        self.state_size = state_size
        self.gamma = 0.97
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.999
        self.batch_size = 96
        self.memory = deque(maxlen=memory_size)
        self.memory_size = memory_size
        self.decay_start = int(memory_size * 0.25)

        self.rng = np.random.default_rng(seed)
        self.py_rng = random.Random(seed + 17)

        self.online = ShallowNet(state_size, self.N_ACTIONS, hidden_size, lr=0.0005, rng=self.rng)
        self.target = ShallowNet(state_size, self.N_ACTIONS, hidden_size, lr=0.0005, rng=self.rng)
        self.target.copy_weights_from(self.online)

        self.target_sync = target_sync
        self.update_count = 0
        self.total_reward = 0.0
        self.reward_history = []
        self.normalizer = RunningNormalizer(state_size)

        self.prev_action = -1
        self.prev_alpha_change = 0.0
        self.hold_conf_threshold = 0.04
        self.alpha_baseline_ema = 0.0

    def build_state(
        self,
        amm_hist,
        mkt_hist,
        sentiment,
        liq_ratio,
        fees_norm,
        volatility,
        dislocation,
        alpha_norm,
        mom_short,
        mom_long,
        reversal_flag,
        range_offset,
        cooldown,
        alpha_drawdown_norm,
    ):
        amm_ret = (
            (amm_hist[-1] - amm_hist[-2]) / amm_hist[-2]
            if len(amm_hist) >= 2 and amm_hist[-2] != 0
            else 0.0
        )
        mkt_ret = (
            (mkt_hist[-1] - mkt_hist[-2]) / mkt_hist[-2]
            if len(mkt_hist) >= 2 and mkt_hist[-2] != 0
            else 0.0
        )
        raw = np.array(
            [
                amm_ret,
                mkt_ret,
                sentiment,
                liq_ratio,
                fees_norm,
                volatility,
                dislocation,
                alpha_norm,
                mom_short,
                mom_long,
                reversal_flag,
                range_offset,
                cooldown,
                alpha_drawdown_norm,
            ],
            dtype=float,
        )
        return self.normalizer.update_and_normalize(raw)

    def act(self, state):
        if self.rng.random() <= self.epsilon:
            # Exploration is intentionally hold-biased to avoid destructive churn.
            probs = np.array([0.55, 0.16, 0.16, 0.065, 0.065], dtype=float)
            return int(self.rng.choice(self.N_ACTIONS, p=probs))

        q = self.online.predict(state)
        ranked = np.argsort(q)
        best = int(ranked[-1])
        second = int(ranked[-2]) if len(ranked) > 1 else best
        confidence_gap = float(q[best] - q[second])

        # Confidence-aware hold gate to reduce noisy churn.
        if best != 0 and confidence_gap < self.hold_conf_threshold:
            if self.rng.random() < 0.75:
                return 0
        return best

    def remember(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def replay(self):
        if len(self.memory) < self.batch_size:
            return
        batch = self.py_rng.sample(list(self.memory), self.batch_size)
        for s, a, r, s2, done in batch:
            tgt = r if done else r + self.gamma * np.max(self.target.predict(s2))
            self.online.update(s, a, tgt)

        self.update_count += 1
        if self.update_count % self.target_sync == 0:
            self.target.copy_weights_from(self.online)

        if len(self.memory) >= self.decay_start and self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def calculate_reward(
        self,
        alpha_change,
        fee_yield,
        rebalance_cost_ratio,
        il_excess,
        drawdown_rise,
        action,
        episode_vol,
    ):
        # Bounded risk-adjusted reward avoids linear collapse and improves
        # cross-regime comparability.
        # Center alpha by an EMA baseline to avoid persistent one-direction drift.
        alpha_adv = alpha_change - self.alpha_baseline_ema
        self.alpha_baseline_ema = 0.98 * self.alpha_baseline_ema + 0.02 * alpha_change

        vol_scale = max(episode_vol, 0.006)
        alpha_norm = alpha_adv / vol_scale

        reward = 2.4 * np.tanh(3.0 * alpha_norm)
        reward += 0.45 * np.tanh(10.0 * fee_yield)
        reward -= 0.55 * np.tanh(100.0 * rebalance_cost_ratio)
        reward -= 0.45 * np.tanh(6.0 * max(0.0, il_excess))
        reward -= 0.40 * np.tanh(6.0 * max(0.0, drawdown_rise))

        # Anti-thrashing penalties.
        if action != 0 and action == self.prev_action:
            reward -= 0.18
        if (self.prev_action, action) in ((1, 2), (2, 1), (3, 4), (4, 3)):
            reward -= 0.12

        if action == 0:
            reward += 0.02

        # Penalize noisy alpha jumps to improve consistency.
        smooth_penalty = abs(alpha_change - self.prev_alpha_change)
        reward -= 0.25 * np.tanh(10.0 * smooth_penalty)

        self.prev_action = action
        self.prev_alpha_change = alpha_change
        self.total_reward += reward
        self.reward_history.append(reward)
        return reward
# Helpers
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


## Section 5: Scenario + Strategy Runners

Generates controlled multi-regime scenarios and defines static/AI strategy execution on identical exogenous paths.


In [10]:
def pool_share(env, lp):
    return min(
        lp.deposit_a / max(env.initial_token_a, 1e-9),
        lp.deposit_b / max(env.initial_token_b, 1e-9),
    )


def rolling_vol(series, window=20):
    if len(series) < 2:
        return 0.01
    recent = series[-min(window, len(series)):]
    rets = [
        (recent[i] - recent[i - 1]) / recent[i - 1]
        for i in range(1, len(recent))
        if recent[i - 1] > 0
    ]
    return float(np.std(rets)) if rets else 0.01


def momentum_return(series, window):
    if len(series) < window + 1:
        return 0.0
    prev = series[-window]
    if prev <= 0:
        return 0.0
    return float((series[-1] - prev) / prev)


def active_fraction(mkt_price, range_center, half_width, liq_ratio):
    if mkt_price <= 0 or range_center <= 0:
        return 0.0
    lo = range_center * (1 - half_width)
    hi = range_center * (1 + half_width)
    if lo <= mkt_price <= hi:
        return float(np.clip(liq_ratio, 0.0, 1.0))
    dist = abs(np.log(max(mkt_price / range_center, 1e-9)))
    decay = np.exp(-8.0 * dist)
    return float(np.clip(liq_ratio * (0.1 + 0.4 * decay), 0.0, 1.0))


def generate_market_path(num_steps, seed, regime_name=None):
    """Generate deterministic market paths across multiple regimes."""
    rng = np.random.default_rng(seed)

    if regime_name is None:
        regime_name = ["trend_up", "trend_down", "mean_revert", "high_vol"][seed % 4]

    price = 1.0
    anchor = 1.0
    path = []

    for t in range(num_steps):
        if regime_name == "trend_up":
            drift = 0.0008
            vol = 0.012
        elif regime_name == "trend_down":
            drift = -0.0008
            vol = 0.012
        elif regime_name == "mean_revert":
            vol = 0.013
            drift = -0.18 * np.log(max(price / max(anchor, 1e-9), 1e-9))
        else:
            drift = 0.0
            vol = 0.028

        if t > 0 and t % 140 == 0:
            vol *= float(rng.choice([0.9, 1.0, 1.15]))

        shock = float(rng.normal(drift, vol))
        price = max(1e-4, price * np.exp(shock))
        path.append(price)

    return path, regime_name


def generate_sentiment_feed(price_feed, seed):
    rng = np.random.default_rng(seed)
    sentiment = []
    s_prev = 0.0

    for i, p in enumerate(price_feed):
        if i == 0:
            ret = 0.0
            local_vol = 0.01
        else:
            prev = price_feed[i - 1]
            ret = (p - prev) / max(prev, 1e-9)
            left = max(0, i - 20)
            recent = price_feed[left : i + 1]
            if len(recent) > 1:
                rets = [
                    (recent[j] - recent[j - 1]) / max(recent[j - 1], 1e-9)
                    for j in range(1, len(recent))
                ]
                local_vol = float(np.std(rets)) if rets else 0.01
            else:
                local_vol = 0.01

        raw = np.tanh(ret / max(local_vol, 0.002) * 3.0) + float(rng.normal(0.0, 0.12))
        s_prev = 0.75 * s_prev + 0.25 * raw
        sentiment.append(float(np.clip(s_prev, -1.0, 1.0)))

    return sentiment


BASE_RANGE_HALF_WIDTH = 0.12


def run_static_strategy(scenario, num_steps):
    env = AMMEnvironment(
        price_feed=list(scenario["price_feed"]),
        rng=np.random.default_rng(scenario["env_seed"]),
    )
    sa = SentimentAnalyzer(sentiment_feed=scenario["sentiment_feed"])
    lp = LiquidityProvider(1000, 1000, env.get_price())
    share = pool_share(env, lp)
    values, il_list, disloc_list = [], [], []

    for _ in range(num_steps):
        s = sa.get_sentiment(env.price_history)
        info = env.simulate_market_activity(s)
        lp.add_fees(info["fees_collected_b"] * share)
        p = env.get_price()
        values.append(lp.calculate_position_value(p, active_exposure=1.0))
        il_list.append(lp.calculate_impermanent_loss(p))
        disloc_list.append(info["price_dislocation"])

    return {
        "values": values,
        "il": il_list,
        "fees": lp.fees_earned,
        "disloc": disloc_list,
        "price_hist": env.price_history,
        "mkt_hist": env.market_price_history,
    }


def run_ai_strategy(scenario, num_steps, agent, static_values, static_il):
    env = AMMEnvironment(
        price_feed=list(scenario["price_feed"]),
        rng=np.random.default_rng(scenario["env_seed"]),
    )
    sa = SentimentAnalyzer(sentiment_feed=scenario["sentiment_feed"])
    lp = LiquidityProvider(1000, 1000, env.get_price())
    share = pool_share(env, lp)

    liq_ratio = 0.90
    range_center = env.market_price
    cum_cost = 0.0

    # Anti-thrashing controls.
    liq_cooldown = 0
    recenter_cooldown = 0

    values, il_list, active_list, actions_list = [], [], [], []
    prev_ai_value = lp.initial_value
    peak_alpha = 0.0
    prev_alpha_dd = 0.0

    for step in range(num_steps):
        s = sa.get_sentiment(env.price_history)
        info = env.simulate_market_activity(s)
        mkt = info["market_price"]
        p = env.get_price()

        vol = rolling_vol(env.market_price_history, 20)
        disloc = info["price_dislocation"]
        mom_short = momentum_return(env.market_price_history, 8)
        mom_long = momentum_return(env.market_price_history, 32)
        reversal_flag = float(
            (np.sign(mom_short) != 0)
            and (np.sign(mom_long) != 0)
            and (np.sign(mom_short) != np.sign(mom_long))
        )

        target_liq = 0.96 - 1.8 * vol - 0.08 * abs(disloc) - 0.05 * reversal_flag
        target_liq = float(np.clip(target_liq, DQLAgent.MIN_LIQ, DQLAgent.MAX_LIQ))

        base_half_width = BASE_RANGE_HALF_WIDTH * (1.0 + 8.0 * vol)
        half_width = float(np.clip(base_half_width * (1.0 + 0.45 * reversal_flag), 0.08, 0.28))

        il_raw = lp.calculate_impermanent_loss(p)
        active = active_fraction(mkt, range_center, half_width, liq_ratio)
        il_now = il_raw * active

        static_val_now = static_values[step] if step < len(static_values) else static_values[-1]
        cur_ai_val = lp.calculate_position_value(p, active, cum_cost)
        alpha_norm = (cur_ai_val - static_val_now) / max(lp.initial_value, 1.0)

        alpha_value = cur_ai_val - static_val_now
        peak_alpha = max(peak_alpha, alpha_value)
        alpha_dd = max(0.0, peak_alpha - alpha_value)

        range_offset = float(np.log(max(mkt / max(range_center, 1e-9), 1e-9)))
        cooldown_norm = min(max(liq_cooldown, recenter_cooldown) / 6.0, 1.0)
        alpha_dd_norm = alpha_dd / max(lp.initial_value, 1.0)

        state = agent.build_state(
            env.price_history,
            env.market_price_history,
            s,
            liq_ratio,
            lp.fees_earned / max(lp.initial_value, 1.0),
            vol,
            disloc,
            alpha_norm,
            mom_short,
            mom_long,
            reversal_flag,
            range_offset,
            cooldown_norm,
            alpha_dd_norm,
        )

        action = agent.act(state)

        # Enforce cooldown for non-hold actions.
        if liq_cooldown > 0 and action in (1, 2):
            action = 0
        if recenter_cooldown > 0 and action in (3, 4):
            action = 0

        cost = 0.0
        if action == 1:
            liq_ratio = min(liq_ratio + 0.05, DQLAgent.MAX_LIQ)
            cost = env.rebalance_gas_cost_b
            liq_cooldown = 3
        elif action == 2:
            liq_ratio = max(liq_ratio - 0.05, DQLAgent.MIN_LIQ)
            cost = env.rebalance_gas_cost_b
            liq_cooldown = 3
        elif action == 3:
            range_center *= 1.015
            cost = env.rebalance_gas_cost_b * 2.0
            recenter_cooldown = 2
        elif action == 4:
            range_center *= 0.985
            cost = env.rebalance_gas_cost_b * 2.0
            recenter_cooldown = 2

        drift_rate = 0.06 if reversal_flag > 0 else 0.025
        range_center = (1.0 - drift_rate) * range_center + drift_rate * mkt

        liq_ratio = float(np.clip(liq_ratio, target_liq - 0.12, target_liq + 0.12))
        liq_ratio = float(np.clip(liq_ratio, DQLAgent.MIN_LIQ, DQLAgent.MAX_LIQ))

        cum_cost += cost
        liq_cooldown = max(0, liq_cooldown - 1)
        recenter_cooldown = max(0, recenter_cooldown - 1)

        active = active_fraction(mkt, range_center, half_width, liq_ratio)
        il_now = il_raw * active

        step_fee = info["fees_collected_b"] * share * active
        lp.add_fees(step_fee)
        new_ai_val = lp.calculate_position_value(p, active, cum_cost)

        static_prev = static_values[step - 1] if step > 0 else lp.initial_value
        alpha_change = ((new_ai_val - static_val_now) - (prev_ai_value - static_prev)) / max(lp.initial_value, 1.0)

        static_il_now = static_il[step] if step < len(static_il) else static_il[-1]
        il_excess = max(0.0, abs(il_now) - abs(static_il_now))

        alpha_value = new_ai_val - static_val_now
        peak_alpha = max(peak_alpha, alpha_value)
        alpha_dd = max(0.0, peak_alpha - alpha_value)
        drawdown_rise = max(0.0, alpha_dd - prev_alpha_dd)
        prev_alpha_dd = alpha_dd

        reward = agent.calculate_reward(
            alpha_change=alpha_change,
            fee_yield=step_fee / max(lp.initial_value, 1.0),
            rebalance_cost_ratio=cost / max(lp.initial_value, 1.0),
            il_excess=il_excess,
            drawdown_rise=drawdown_rise,
            action=action,
            episode_vol=vol,
        )

        next_alpha_norm = (new_ai_val - static_val_now) / max(lp.initial_value, 1.0)
        next_dd_norm = alpha_dd / max(lp.initial_value, 1.0)
        next_state = agent.build_state(
            env.price_history,
            env.market_price_history,
            s,
            liq_ratio,
            lp.fees_earned / max(lp.initial_value, 1.0),
            vol,
            disloc,
            next_alpha_norm,
            mom_short,
            mom_long,
            reversal_flag,
            range_offset,
            min(max(liq_cooldown, recenter_cooldown) / 6.0, 1.0),
            next_dd_norm,
        )

        done = (step == num_steps - 1)
        agent.remember(state, action, reward, next_state, done)
        agent.replay()

        prev_ai_value = new_ai_val
        values.append(new_ai_val)
        il_list.append(il_now)
        active_list.append(active)
        actions_list.append(action)

    return {
        "values": values,
        "il": il_list,
        "fees": lp.fees_earned,
        "active": active_list,
        "actions": actions_list,
        "disloc": env.dislocation_history[1:],
        "price_hist": env.price_history,
        "mkt_hist": env.market_price_history,
    }


## Section 6: Simulation Orchestrator

Runs multi-episode comparison, prints summary metrics, and saves diagnostics plots.


In [11]:
class AIAMMSimulation:
    def __init__(self, num_steps=1000, num_episodes=160, seed=42):
        self.num_steps = int(num_steps)
        self.num_episodes = int(num_episodes)
        self.base_seed = int(seed)
        self.agent = DQLAgent(seed=seed)

    def _build_episode_scenario(self, episode_idx):
        seed_base = self.base_seed + episode_idx * 1009
        regimes = ["trend_up", "trend_down", "mean_revert", "high_vol"]
        regime = regimes[episode_idx % len(regimes)]

        price_feed, regime_used = generate_market_path(
            num_steps=self.num_steps,
            seed=seed_base,
            regime_name=regime,
        )
        sentiment_feed = generate_sentiment_feed(price_feed, seed_base + 13)

        return {
            "price_feed": price_feed,
            "sentiment_feed": sentiment_feed,
            "env_seed": seed_base + 23,
            "regime": regime_used,
        }

    def run_comparison(self):
        print("=" * 76)
        print("  AIAMM V8 FRA: Regime-Aware, Controlled RL Liquidity Management")
        print("=" * 76)
        print(f"  Episodes: {self.num_episodes} | Steps/episode: {self.num_steps} | Seed: {self.base_seed}")
        print("-" * 76)

        s_vals_ep, ai_vals_ep = [], []
        s_il_ep, ai_il_ep = [], []
        s_fees_ep, ai_fees_ep = [], []
        alpha_ep = []
        regime_alpha = {"trend_up": [], "trend_down": [], "mean_revert": [], "high_vol": []}

        last_static = None
        last_ai = None

        for ep in range(self.num_episodes):
            scenario = self._build_episode_scenario(ep)

            static = run_static_strategy(scenario, self.num_steps)
            ai = run_ai_strategy(scenario, self.num_steps, self.agent, static["values"], static["il"])

            alpha = ai["values"][-1] - static["values"][-1]

            s_vals_ep.append(static["values"][-1])
            ai_vals_ep.append(ai["values"][-1])
            s_il_ep.append(np.mean(static["il"]) * 100)
            ai_il_ep.append(np.mean(ai["il"]) * 100)
            s_fees_ep.append(static["fees"])
            ai_fees_ep.append(ai["fees"])
            alpha_ep.append(alpha)
            regime_alpha[scenario["regime"]].append(alpha)

            last_static = static
            last_ai = ai

            if (ep + 1) % 15 == 0:
                a = np.array(alpha_ep, dtype=float)
                med = float(np.median(a)) if len(a) else 0.0
                print(
                    f"  Ep {ep+1:>3}/{self.num_episodes}"
                    f" | alpha_mean={np.mean(a):+8.2f}"
                    f" | alpha_med={med:+8.2f}"
                    f" | win={np.mean(a > 0) * 100:5.1f}%"
                    f" | eps={self.agent.epsilon:.3f}"
                    f" | mem={len(self.agent.memory)}"
                )

        alpha_arr = np.array(alpha_ep, dtype=float)

        print()
        print("=" * 76)
        print("  RESULTS SUMMARY (controlled same-path episodes)")
        print("=" * 76)
        print(f"{'Metric':<42} {'Static':>10} {'AI':>10} {'Delta':>11}")
        print("-" * 76)

        sv = float(np.mean(s_vals_ep))
        av = float(np.mean(ai_vals_ep))
        print(f"{'Avg Final Value ($)':<42} {sv:>10.2f} {av:>10.2f} {av-sv:>+11.2f}")

        si = float(np.mean(s_il_ep))
        ai_i = float(np.mean(ai_il_ep))
        print(f"{'Avg IL (%)':<42} {si:>9.3f}% {ai_i:>9.3f}% {ai_i-si:>+10.3f}%")

        sf = float(np.mean(s_fees_ep))
        af = float(np.mean(ai_fees_ep))
        print(f"{'Avg Fees ($)':<42} {sf:>10.2f} {af:>10.2f} {af-sf:>+11.2f}")

        alpha_mean = float(np.mean(alpha_arr)) if len(alpha_arr) else 0.0
        alpha_median = float(np.median(alpha_arr)) if len(alpha_arr) else 0.0
        alpha_std = float(np.std(alpha_arr)) if len(alpha_arr) else 0.0
        sharpe = alpha_mean / (alpha_std + 1e-9)
        win_rate = float(np.mean(alpha_arr > 0) * 100) if len(alpha_arr) else 0.0
        p10 = float(np.percentile(alpha_arr, 10)) if len(alpha_arr) else 0.0
        p90 = float(np.percentile(alpha_arr, 90)) if len(alpha_arr) else 0.0

        print(f"{'Alpha mean ($)':<42} {'-':>10} {alpha_mean:>10.2f} {'':>11}")
        print(f"{'Alpha median ($)':<42} {'-':>10} {alpha_median:>10.2f} {'':>11}")
        print(f"{'Alpha p10 / p90 ($)':<42} {'-':>10} {p10:>5.2f}/{p90:<5.2f} {'':>11}")
        print(f"{'Alpha std ($)':<42} {'-':>10} {alpha_std:>10.2f} {'':>11}")
        print(f"{'Alpha Sharpe':<42} {'-':>10} {sharpe:>10.3f} {'':>11}")
        print(f"{'Win rate (%)':<42} {'-':>10} {win_rate:>9.1f}% {'':>11}")

        print()
        print("  Regime alpha means:")
        for regime in ["trend_up", "trend_down", "mean_revert", "high_vol"]:
            vals = np.array(regime_alpha[regime], dtype=float)
            val = float(np.mean(vals)) if len(vals) else 0.0
            print(f"    {regime:<12} {val:+9.2f}")

        print()
        print("  Action distribution (last episode):")
        acts = last_ai["actions"]
        for i, name in enumerate(DQLAgent.ACTION_NAMES):
            c = acts.count(i)
            pct = c / len(acts) * 100
            bar = "#" * int(max(1, pct / 2))
            print(f"    {name:<26} {c:>5} ({pct:>5.1f}%) {bar}")

        print(
            f"\n  Final epsilon={self.agent.epsilon:.4f}"
            f" | total_reward={self.agent.total_reward:.2f}"
            f" | updates={self.agent.update_count}"
        )

        self._plot(last_static, last_ai, alpha_arr)

        return {
            "static_value": sv,
            "ai_value": av,
            "alpha_mean": alpha_mean,
            "alpha_median": alpha_median,
            "alpha_sharpe": sharpe,
            "win_rate": win_rate,
            "alpha_p10": p10,
            "alpha_p90": p90,
            "static_il": si,
            "ai_il": ai_i,
            "static_fees": sf,
            "ai_fees": af,
        }

    def _plot(self, static, ai, alpha_arr):
        fig, axes = plt.subplots(2, 3, figsize=(19, 11))
        fig.suptitle(
            "AIAMM V8 FRA: Controlled Regime-Aware Comparison",
            fontsize=15,
            fontweight="bold",
        )

        steps = range(self.num_steps)

        # 1) Portfolio value
        ax = axes[0, 0]
        ax.plot(steps, static["values"], label="Static LP", color="steelblue", alpha=0.85)
        ax.plot(steps, ai["values"], label="AI Agent", color="seagreen", alpha=0.85)
        ax.axhline(static["values"][0], color="gray", linestyle=":", lw=0.8)
        ax.set_title("Portfolio Value (same price path)")
        ax.set_xlabel("Step")
        ax.set_ylabel("$")
        ax.legend()
        ax.grid(True, alpha=0.3)

        # 2) Per-step alpha
        ax = axes[0, 1]
        a_series = [ai["values"][i] - static["values"][i] for i in range(self.num_steps)]
        ax.plot(steps, a_series, color="darkorange", lw=0.9, alpha=0.9)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_title("Alpha: AI - Static (per step)")
        ax.set_xlabel("Step")
        ax.set_ylabel("Alpha ($)")
        ax.grid(True, alpha=0.3)

        # 3) IL comparison
        ax = axes[0, 2]
        ax.plot(steps, [v * 100 for v in static["il"]], label="Static IL", color="tomato", alpha=0.8)
        ax.plot(steps, [v * 100 for v in ai["il"]], label="AI IL", color="darkorange", alpha=0.8)
        ax.set_title("Impermanent Loss")
        ax.set_xlabel("Step")
        ax.set_ylabel("IL (%)")
        ax.legend()
        ax.grid(True, alpha=0.3)

        # 4) Action distribution
        ax = axes[1, 0]
        counts = [ai["actions"].count(i) for i in range(DQLAgent.N_ACTIONS)]
        colors = ["#888", "#2ca02c", "#d62728", "#1f77b4", "#ff7f0e"]
        bars = ax.bar(range(DQLAgent.N_ACTIONS), counts, color=colors, alpha=0.8)
        for bar, c in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2, str(c), ha="center", va="bottom", fontsize=9)
        ax.set_xticks(range(DQLAgent.N_ACTIONS))
        ax.set_xticklabels(["Hold", "Inc", "Dec", "Up", "Down"], rotation=0)
        ax.set_title("Action Distribution (last episode)")
        ax.set_ylabel("Count")
        ax.grid(True, alpha=0.3, axis="y")

        # 5) Cumulative reward
        ax = axes[1, 1]
        if self.agent.reward_history:
            cum_r = np.cumsum(self.agent.reward_history)
            ax.plot(cum_r, color="seagreen", alpha=0.85, lw=0.8)
            ax.axhline(0, color="black", lw=0.8)
        ax.set_title("Agent Cumulative Reward")
        ax.set_xlabel("Training step")
        ax.set_ylabel("Sum reward")
        ax.grid(True, alpha=0.3)

        # 6) Alpha distribution
        ax = axes[1, 2]
        ax.hist(alpha_arr, bins=25, color="darkorange", alpha=0.75, edgecolor="white")
        if len(alpha_arr):
            ax.axvline(0, color="black", lw=1.2, ls="--", label="Break-even")
            ax.axvline(np.mean(alpha_arr), color="seagreen", lw=1.5, label=f"Mean={np.mean(alpha_arr):+.2f}")
            ax.axvline(np.median(alpha_arr), color="tab:blue", lw=1.4, label=f"Median={np.median(alpha_arr):+.2f}")
            win = np.mean(alpha_arr > 0) * 100
            ax.set_title(f"Alpha Distribution - Win Rate {win:.0f}%")
            ax.legend()
        else:
            ax.set_title("Alpha Distribution")
        ax.set_xlabel("Alpha ($)")
        ax.set_ylabel("Episodes")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("aiamm_v8fra_resultsA.png", dpi=150, bbox_inches="tight")
        print("\nSaved: aiamm_v8fra_resultsA.png")
# Main
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


## Section 7: Main Execution

Entry point for end-to-end execution.


In [ ]:
def main():
    print()
    print("+" + "-" * 74 + "+")
    print("| AIAMM V8 FRA: controlled same-path RL with regime-aware risk controls      |")
    print("| reward: bounded relative alpha + risk terms, anti-thrash cooldowns         |")
    print("+" + "-" * 74 + "+")
    print()
    sim = AIAMMSimulation(num_steps=1000, num_episodes=160, seed=42)
    results = sim.run_comparison()
    print("\nFinal results:")
    print(results)


if __name__ == "__main__":
    main()



+--------------------------------------------------------------------------+
| AIAMM V8 FRA: controlled same-path RL with regime-aware risk controls      |
| reward: bounded relative alpha + risk terms, anti-thrash cooldowns         |
+--------------------------------------------------------------------------+

  AIAMM V8 FRA: Regime-Aware, Controlled RL Liquidity Management
  Episodes: 160 | Steps/episode: 1000 | Seed: 42
----------------------------------------------------------------------------
  Ep  15/160 | alpha_mean=   -7.40 | alpha_med=  -26.26 | win= 20.0% | eps=0.050 | mem=12000
  Ep  30/160 | alpha_mean=  -11.53 | alpha_med=  -22.94 | win= 16.7% | eps=0.050 | mem=12000
  Ep  45/160 | alpha_mean=  -12.43 | alpha_med=  -24.93 | win= 17.8% | eps=0.050 | mem=12000
